## A simple Lasso model model to predict property prices 

I just wanted to share my first attempt at creating a predictive model in Python. Hopefully, you will find the notebook useful and can adapt it to your own regression projects.

### Background on property valuations  

Before with dive into the data analysis, let's quickly recap how we would derive the value of a property 

**Property value = property size in sqft $\times$ property cost per sqft  $\times$ adjustments**, 

which is equivalent to

**Log (property value) = Log (living area in sqft) + Log(cost per sqft) + $\sum$  Log (adjustments)**

Thus, a linear model to predict the log of the property might be worth a try ...

Let's get started!

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold, cross_val_score

train = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("../input/house-prices-advanced-regression-techniques/test.csv")

train["Test"]=0
test["Test"]=1
X_all = pd.concat((train, test),sort=True).reset_index(drop=True)
X_all.loc[:,"SalePrice"] = np.log(X_all["SalePrice"])

## Preprocssing

### Explore and understand the data

This is the key first step in any data science project. I know that this will take quite some time and effort but it is invaluable. There are excellent dicussions and visualisations in other notebooks - please have look! It will help you to become familiar with the data.  

### Missing data

I decided to simply fill in missing values with the most typical value for each feature. Have a look at some of the other kernels that contain more in-depth explanations for this step. 

In [2]:
summary = pd.DataFrame({"Missing":len(X_all)- X_all.count(),"Typical":np.nan})

for col in summary[(summary.Missing > 0)].index: 
    if col == "SalePrice": continue
    colmode=X_all[col].mode()
    summary.loc[col,"Typical"]=colmode[0]
    X_all[col].fillna(colmode[0],inplace=True)

summary[summary.Missing > 0]

,Missing,Typical
Alley,2721,Grvl
BsmtCond,82,TA
BsmtExposure,82,No
BsmtFinSF1,1,0
BsmtFinSF2,1,0
BsmtFinType1,79,Unf
BsmtFinType2,80,Unf
BsmtFullBath,2,0
BsmtHalfBath,2,0
BsmtQual,81,TA


### Analyse and remove outliers

As identified in the original paper, the sale price for some of the new homes does not reflect the full market value. Two large and two smaller properties appear to be priced significantly below market value. In addition, there is one property with an abnormally high price. I decided to remove these properties from the train dataset.

In [3]:
outliers=[]
train=X_all[X_all.Test==0].copy()
hover=["Id","GrLivArea","OverallCond","MSZoning","YearBuilt","YearRemodAdd","Functional","SaleCondition","SaleType","BldgType","Neighborhood","LotArea"]
train["LogSF"]=np.log(train.GrLivArea)
fig = px.scatter(train[train.SaleType=="New"],y='SalePrice', x='LogSF',color="OverallQual",size="OverallCond",
           hover_data=hover)
fig.show()
outliers+=[524,1299,1325,49,689]

Now, let's have a look at outliers for the pro-owned homes. A small number of properties are classed as commercial. We will take a deeper look at these to help us confirm the most significant outliers. 

In [4]:
fig = px.scatter(train[train.SaleType!="New"],y='SalePrice', x='LogSF',color="MSZoning",size="OverallQual",
           hover_data=hover)
fig.show()

fig = px.scatter(train[(train.MSZoning=="C (all)") & (train.SaleType!="New")],y='SalePrice', x='LogSF',color="SaleCondition",size="OverallQual",
           hover_data=hover,trendline="ols")
fig.show()
outliers+=[496,31,969]

## Feature engineering

As the number of samples is very small in relation to the number of features, I summarised the most relevant categorical features as numerical. This method should also limit the effect of outliers/gaps in the train dataset on the prediction.

The calibration for the weights in the mapping tables was estimated by fitting a regression model that includes the categorical features - you can find an example of this appraoch at the end of the notebook.

In [5]:
functional_map = {"Sal":1,"Sev":0.3,"Maj1":0.1,"Maj2":0.15,"Mod":0.1,"Min1":0.05,"Min2":0.05,"Typ":0} 
quality_map = {"Ex":5,"Gd":4,"TA":3,"Fa":2,"Po":1,"NA":0,0:0}
exposure_map = {"Gd":3,"Av":1,"Mn":0,"No":0,"NA":0}
lotqual_map = {"CulDSac":2,"Inside":0,"Corner":1,"FR2": -1,"FR3":-2}


def feature_eng (X,num_to_categ={},full_data=False):    
     
    # My preference has been to identify a small subset of features that will result in a good fit.
    # However, including all features (set the parameter full_data = True) will slighlty improve 
    # CV & Kaggle scores and the final run for the submission will include all features.
    
    new_columns= ["SF","BaseGood","BaseTA","Age","Pool_Ind","New","SaleCondition_Abn","Res","Pred"]
    for el in new_columns: X.insert(1,el,0)
    
    # Create summary indicators
    
    X.loc[(X.SaleType=="New") & (X.YearBuilt >= X.YrSold-1),"New"]= 1
    X.loc[X.SaleCondition == "Abnorml","SaleCondition_Abn"]=1
    X.loc[(X.PoolArea > 0),"Pool_Ind"]= 1
    
    # Summarise categorical features as numerical
    
    qual_cols = ["ExterQual","ExterCond","BsmtQual","BsmtCond","HeatingQC","KitchenQual","FireplaceQu","GarageQual"]
    for col in qual_cols: X.loc[:,col]= X[col].map(quality_map)
        
    X["LotConfig"]=X["LotConfig"].map(lotqual_map)
    X["Functional"]=X["Functional"].map(functional_map)
    X["Age"]= np.maximum((X["YrSold"]-X["YearBuilt"])/10,0)
    X["QualityScore"]= (X.KitchenQual * 2.5 + X.FireplaceQu *1 + X.GarageQual* 1.5 + X.HeatingQC * 1.5 + 2* X.BsmtQual)/8.5-3
    X["OverallCond"]= (X.OverallCond-5) 
    X["OverallQual"]= (X.OverallQual-5) 
    
    # Adjust total living area for above and below ground living areas based on quality:
    # GrLivArea capures only the above ground surface 
    # whilst some properties include significant good quality below ground living areas
    # First ground living space appears to be valued slightly higher than second ground
    
    X.loc[X.BsmtFinType1.isin({"GLQ","ALQ"}),"BaseGood"]+= X.loc[X.BsmtFinType1.isin({"GLQ","ALQ"}),"BsmtFinSF1"]
    X.loc[X.BsmtFinType1.isin({"BLQ","Rec","LwQ"}),"BaseTA"]+= X.loc[X.BsmtFinType1.isin({"BLQ","Rec","LwQ"}),"BsmtFinSF1"]
    X.loc[X.BsmtFinType2.isin({"GLQ","ALQ"}),"BaseGood"]+= X.loc[X.BsmtFinType2.isin({"GLQ","ALQ"}),"BsmtFinSF2"]
    X.loc[X.BsmtFinType2.isin({"BLQ","Rec","LwQ"}),"BaseTA"]+= X.loc[X.BsmtFinType2.isin({"BLQ","Rec","LwQ"}),"BsmtFinSF2"]
    X["SF"]=  X.GrLivArea - 0.1*X["2ndFlrSF"]+(X.BaseGood*.65+X.BaseTA*0.5)+0.25*X.BsmtUnfSF
    
    # Give additional credit for good exposure of the basement area
    
    X["BsmtExposure"]= X["BsmtExposure"].map(exposure_map)*((X.BaseGood+X.BaseTA)/X.SF)
    
    # Apply log transform to surface areas
    
    X["SFLog"]=np.log(X.SF)
    X["LotArea"]=np.log(X.LotArea) - np.log(X.SF)
    
    
    # Select features to include in the model (for the final run we will include all)
    
    if not full_data:
        X=X[["Res","Pred","SalePrice","Test","Id"] + ["SFLog","Functional","OverallQual","OverallCond",
            "LotArea","GarageCars","MSZoning","Neighborhood","New","Age","BldgType","SaleCondition_Abn",
            "Pool_Ind","BsmtExposure","QualityScore","LotConfig","Condition1"]]
          
    X = pd.get_dummies(X,columns=list(num_to_categ))
    X = pd.get_dummies(X)
    
    # Correct for overfit:
    # I decided to exclude all features where fewer than 5 samples differ from the mode 
    
    for el in set(X.columns).difference({"Test","Pred","Res"}):
        col_mode=X.loc[X.Test==0,el].mode()[0]
        if len(X.loc[(X.Test == 0) & (X[el] !=col_mode),"SFLog"]) < 5: 
                X.drop(columns=el, inplace = True)
                print ("Feature removed:",el)
            
    # Add interaction terms for key features, e.g.
    # overall quality tends to increase with surface area
    # overall condition is related to property age (all new properties are rated 5)
  
    inter_features ={"SFLog","Age","OverallCond","OverallQual","QualityScore","Functional"}.difference(num_to_categ) 
    poly = PolynomialFeatures(interaction_only=True,include_bias=False) 
    inter_cols = poly.fit_transform(X[inter_features])
    X1= pd.DataFrame(inter_cols,columns= poly.get_feature_names(list(inter_features)),index=X.index)                            
    X = pd.concat([X1,X.drop(columns=inter_features)],axis=1)
    
    return X

## Modelling

Let's go ahead and fit our regression model. 

In [6]:
def model_fit(model,X,target_col,folds=5):
    
    kf = KFold(folds, shuffle=True, random_state=4991)
    
    drop_cols = ["Test","Id","Pred","Res",target_col]
    X_train = X[X.Test==0].drop(columns=drop_cols)
    y_train = X.loc[X.Test==0,target_col]
    model.fit(X_train,y_train)
    X["Pred"] = model.predict(X.drop(columns=drop_cols))
    X["Res"]= X.Pred-X[target_col]
    
    score =  (-cross_val_score(model, X_train, y_train, scoring="neg_mean_squared_error", cv = kf))**0.5
    df = pd.DataFrame({"Coeff":model.coef_,"Y mean":(X_train*model.coef_).mean(),"Y std":(X_train*model.coef_).std()}).sort_values("Y std",ascending=False)
    return X, df, score

X=X_all[~X_all.Id.isin(outliers)].copy()
model = Lasso(alpha=0.0003,random_state=3591,max_iter=100000,fit_intercept=True)

X = feature_eng(X,full_data=False)
X, df, score = model_fit(model,X,"SalePrice")
print("\nCV score: ",score,"\nMean: {:.4f} Std: {:.4f}\n".format(score.mean(), score.std()))
df.head(10)

Feature removed: Neighborhood_Blueste
Feature removed: Condition1_RRNe

CV score:  [0.10006875 0.10674617 0.09179433 0.11613134 0.09898907] 
Mean: 0.1027 Std: 0.0082



,Coeff,Y mean,Y std
SFLog,0.672633,5.050243,0.208011
SFLog OverallQual,0.010724,0.091362,0.113624
Age SFLog,-0.003814,-0.103284,0.085105
OverallQual,-0.053730,-0.058985,0.073702
OverallQual QualityScore,0.030628,0.032419,0.047674
LotArea,0.079806,0.127788,0.037807
SFLog OverallCond,0.004408,0.018954,0.036519
GarageCars,0.046799,0.082801,0.034787
Age OverallCond,0.003704,0.012557,0.028825
SFLog QualityScore,0.007284,0.031378,0.025250


The most significant adjustments to property values relate to age, quality and condition of the property, as well as to lot and garage size. 

This feels all very reasonable. 

A a final step, we will check the residuals to detect any additional outliers.   

In [7]:
hover=["Id","OverallCond","Functional","SaleCondition_Abn","LotArea"]
fig = px.scatter(X,y="Res", x='SFLog',color="OverallQual",size="Age",hover_data=hover)
fig.show()

I decided to remove properties with residuals greater than +0.4 as the property values for these data points are not explained by the model. 

In [8]:
outliers+=[589,633,463,813,1433,411,804]
X=X_all[~X_all.Id.isin(outliers)].copy()
X = feature_eng(X)
X, df, score = model_fit(model,X,"SalePrice")
print("\nModel RMSLE CV score: ",score,"\nMean: {:.4f} Std: {:.4f}\n".format(score.mean(), score.std()))
df.head(10)

Feature removed: Neighborhood_Blueste
Feature removed: Condition1_RRNe

Model RMSLE CV score:  [0.09404468 0.09399093 0.0952846  0.09511044 0.09010671] 
Mean: 0.0937 Std: 0.0019



,Coeff,Y mean,Y std
SFLog,0.670951,5.037819,0.207292
SFLog OverallQual,0.011006,0.093911,0.116485
Age SFLog,-0.003902,-0.105594,0.087181
OverallQual,-0.053847,-0.059213,0.073790
OverallQual QualityScore,0.029548,0.031301,0.045893
LotArea,0.081476,0.130333,0.038614
Age OverallCond,0.004458,0.015155,0.034711
GarageCars,0.045518,0.080577,0.033831
SFLog OverallCond,0.003305,0.014247,0.027341
Neighborhood_Crawfor,0.119909,0.004232,0.022134


Finally, we re-run the model with all features included. This approach does slightly improve the CV (and submission) score but will be  difficult explain as many of the features are correlated. I am not sure that I would prefer it over the initial model.  

In [9]:
model = Lasso(alpha=0.00033,random_state=3591,max_iter=100000,fit_intercept=True)
X=X_all[~X_all.Id.isin(outliers)].copy()
X = feature_eng(X,full_data=True)
X, df, score = model_fit(model,X,"SalePrice")
print("\nModel RMSLE CV score: ",score,"\nMean: {:.4f} Std: {:.4f}\n".format(score.mean(), score.std()))
df.head(10)

Feature removed: Condition2_RRNn
Feature removed: Condition2_Artery
Feature removed: Exterior2nd_CBlock
Feature removed: RoofMatl_Metal
Feature removed: PoolQC_Ex
Feature removed: PoolQC_Fa
Feature removed: Condition2_PosN
Feature removed: Condition1_RRNe
Feature removed: Condition2_RRAe
Feature removed: SaleType_Con
Feature removed: MiscFeature_Othr
Feature removed: RoofMatl_Roll
Feature removed: Electrical_Mix
Feature removed: Utilities_AllPub
Feature removed: Heating_Wall
Feature removed: GarageCond_Ex
Feature removed: SaleType_CWD
Feature removed: Neighborhood_Blueste
Feature removed: Heating_OthW
Feature removed: RoofMatl_Membran
Feature removed: Exterior1st_Stone
Feature removed: MiscFeature_TenC
Feature removed: Condition2_PosA
Feature removed: PoolQC_Gd
Feature removed: Exterior1st_CBlock
Feature removed: Electrical_FuseP
Feature removed: RoofStyle_Shed
Feature removed: Exterior1st_AsphShn
Feature removed: Heating_Floor
Feature removed: SaleType_Oth
Feature removed: Foundation_

,Coeff,Y mean,Y std
YearBuilt,0.006775,13.355655,0.204076
Age SFLog,0.005733,0.155145,0.128091
SFLog,0.310137,2.328657,0.095818
SF,0.000114,0.218535,0.068830
2ndFlrSF,0.000102,0.035213,0.044245
OverallQual QualityScore,0.025672,0.027194,0.039872
SFLog OverallQual,0.003045,0.025983,0.032229
LotArea,0.067611,0.108153,0.032042
SFLog QualityScore,0.008881,0.038323,0.030695
GrLivArea,-0.000060,-0.091289,0.030648


### Submission

In [10]:
sub = pd.DataFrame({"Id":X.loc[X.Test==1,"Id"],"SalePrice":np.exp(X.loc[X.Test==1,"Pred"])})
sub.to_csv('submissionV1.1.csv',index=False)

### Conversion of categoricial features - Example

As promised, let's have a look at the functional deductions to check that the mapping weights are reasonable. 

In [11]:
X=X_all[~X_all.Id.isin(outliers)].copy()
X = feature_eng(X,["Functional"])
X, df, score = model_fit(model,X,"SalePrice")
df.loc[["Functional_0.0","Functional_0.05","Functional_0.15","Functional_0.1"],"Coeff"]-df.loc["Functional_0.0","Coeff"]

Feature removed: Neighborhood_Blueste
Feature removed: Condition1_RRNe
Feature removed: Functional_0.3


Functional_0.0     0.000000
Functional_0.05   -0.057487
Functional_0.15   -0.122862
Functional_0.1    -0.085760
Name: Coeff, dtype: float64